# Perceptrón Multicapa en MNIST
### Desde cero (NumPy) y con GPU (PyTorch)

**Arquitectura:** 784 → 256 → 128 → 10

| Parte | Framework | Dispositivo | Objetivo |
|-------|-----------|-------------|----------|
| 1 | NumPy puro | CPU | Entender forward/backward |
| 2 | PyTorch | CPU + GPU | Ver el salto de rendimiento |

---
**Notación que usaremos:**
- $a^{[l]}$ : activaciones de la capa $l$
- $z^{[l]} = a^{[l-1]} W^{[l]} + b^{[l]}$ : preactivaciones
- $\delta^{[l]}$ : gradiente de la pérdida respecto a $z^{[l]}$

## 0. Librerías y datos

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# ── Datos ──────────────────────────────────────────────────────────────────
print('Cargando MNIST...')
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
X, y = mnist.data / 255.0, mnist.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=10_000, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

ModuleNotFoundError: No module named 'sklearn'

---
## Parte 1 — MLP desde cero con NumPy (CPU)

### Forward pass
$$z^{[l]} = a^{[l-1]}\,W^{[l]} + b^{[l]}, \qquad a^{[l]} = \sigma(z^{[l]})$$

Capa de salida usa **Softmax**: $\hat{y}_k = \dfrac{e^{z_k}}{\sum_j e^{z_j}}$

### Backward pass (regla de la cadena)
$$\delta^{[L]} = \hat{y} - y \qquad \text{(Softmax + Cross-entropy, derivada exacta)}$$
$$\delta^{[l]} = \left(\delta^{[l+1]}\,{W^{[l+1]}}^\top\right) \odot \underbrace{a^{[l]}(1-a^{[l]})}_{\sigma'(z^{[l]})}$$
$$\nabla W^{[l]} = \frac{1}{m}\,{a^{[l-1]}}^\top\,\delta^{[l]}, \qquad \nabla b^{[l]} = \frac{1}{m}\sum \delta^{[l]}$$

In [ ]:
class MLP_NumPy:
    """MLP con forward/backward explícito. Sin frameworks."""

    def __init__(self, dims):
        """dims: lista de tamaños de capas, ej. [784, 256, 128, 10]"""
        # Xavier initialization: escala por √(1/fan_in)
        self.W = [np.random.randn(dims[i], dims[i+1]) * np.sqrt(1/dims[i])
                  for i in range(len(dims)-1)]
        self.b = [np.zeros((1, dims[i+1]))
                  for i in range(len(dims)-1)]

    # ── Activaciones ───────────────────────────────────────────────────────
    @staticmethod
    def sigmoid(z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    @staticmethod
    def softmax(z):
        e = np.exp(z - z.max(axis=1, keepdims=True))  # estabilidad numérica
        return e / e.sum(axis=1, keepdims=True)

    # ── Forward ────────────────────────────────────────────────────────────
    def forward(self, X):
        self.a = [X]                        # guardamos a[0] = entrada
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.a[-1] @ W + b
            is_last = (i == len(self.W) - 1)
            self.a.append(self.softmax(z) if is_last else self.sigmoid(z))
        return self.a[-1]                   # ŷ con forma (m, 10)

    # ── Pérdida ────────────────────────────────────────────────────────────
    def cross_entropy(self, y_pred, y_true):
        Y = np.eye(10)[y_true]              # one-hot (m, 10)
        return -np.mean(np.sum(Y * np.log(y_pred + 1e-8), axis=1))

    # ── Backward ───────────────────────────────────────────────────────────
    def backward(self, y_true, lr):
        m   = len(y_true)
        Y   = np.eye(10)[y_true]            # one-hot

        # δ de la capa de salida (Softmax + CE se simplifica a ŷ - y)
        delta = self.a[-1] - Y              # forma (m, 10)

        for i in reversed(range(len(self.W))):
            # Gradientes de pesos y bias
            dW = self.a[i].T @ delta / m
            db = delta.mean(axis=0, keepdims=True)

            # Propagar δ hacia atrás (solo si hay capas anteriores)
            if i > 0:
                da    = delta @ self.W[i].T
                delta = da * self.a[i] * (1 - self.a[i])  # σ'(z) = a(1-a)

            # Actualización de parámetros (Gradient Descent)
            self.W[i] -= lr * dW
            self.b[i] -= lr * db

    # ── Entrenamiento ──────────────────────────────────────────────────────
    def fit(self, X, y, epochs=20, batch_size=256, lr=0.1, X_val=None, y_val=None):
        historia = {'loss': [], 'acc': [], 'val_acc': []}
        m        = len(X)

        for epoch in range(1, epochs+1):
            # Shuffle
            idx = np.random.permutation(m)
            epoch_loss = []

            for start in range(0, m, batch_size):
                Xb = X[idx[start:start+batch_size]]
                yb = y[idx[start:start+batch_size]]
                y_hat = self.forward(Xb)
                epoch_loss.append(self.cross_entropy(y_hat, yb))
                self.backward(yb, lr)

            # Métricas de época
            loss = np.mean(epoch_loss)
            acc  = self.accuracy(X, y)
            historia['loss'].append(loss)
            historia['acc'].append(acc)

            if X_val is not None:
                val_acc = self.accuracy(X_val, y_val)
                historia['val_acc'].append(val_acc)
                print(f'Época {epoch:2d}  loss={loss:.4f}  acc={acc:.4f}  val_acc={val_acc:.4f}')
            else:
                print(f'Época {epoch:2d}  loss={loss:.4f}  acc={acc:.4f}')

        return historia

    def accuracy(self, X, y):
        y_hat = self.forward(X)
        return np.mean(np.argmax(y_hat, axis=1) == y)

In [ ]:
# ── Entrenar MLP NumPy ─────────────────────────────────────────────────────
np.random.seed(42)
mlp_np = MLP_NumPy([784, 256, 128, 10])

t0 = time.time()
hist_np = mlp_np.fit(
    X_train, y_train,
    epochs=20, batch_size=256, lr=0.1,
    X_val=X_test, y_val=y_test
)
t_numpy = time.time() - t0
print(f'\nTiempo total NumPy (CPU): {t_numpy:.1f}s')

In [ ]:
# ── Curvas de aprendizaje ─────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(hist_np['loss'], 'b-o', markersize=4, label='Train')
ax1.set(title='Pérdida (Cross-Entropy)', xlabel='Época', ylabel='Loss')
ax1.grid(alpha=.3); ax1.legend()

ax2.plot(hist_np['acc'],     'b-o', markersize=4, label='Train')
ax2.plot(hist_np['val_acc'], 'r-o', markersize=4, label='Test')
ax2.set(title='Accuracy', xlabel='Época', ylabel='Accuracy')
ax2.grid(alpha=.3); ax2.legend()

plt.suptitle('MLP NumPy (CPU)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Parte 2 — MLP con PyTorch (CPU + GPU)

La misma arquitectura, ahora con diferenciación automática y soporte GPU.

**Clave:** `.to(device)` mueve tensores y modelo entre CPU y GPU.
PyTorch calcula los gradientes automáticamente con `loss.backward()`.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# ── Detectar dispositivo ──────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Convertir datos a tensores ────────────────────────────────────────────
def make_loader(X, y, batch_size=256, shuffle=True):
    tx = torch.tensor(X, dtype=torch.float32)
    ty = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(tx, ty), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_train, y_train)
test_loader  = make_loader(X_test,  y_test,  shuffle=False)

In [ ]:
class MLP_PyTorch(nn.Module):
    """Misma arquitectura que MLP_NumPy, con autograd y soporte GPU."""

    def __init__(self, dims):
        super().__init__()
        capas = []
        for i in range(len(dims)-1):
            capas.append(nn.Linear(dims[i], dims[i+1]))
            if i < len(dims) - 2:          # Sigmoid en capas ocultas
                capas.append(nn.Sigmoid())
        self.red = nn.Sequential(*capas)   # la última capa es lineal (CrossEntropyLoss incluye Softmax)

    def forward(self, x):
        return self.red(x)

In [ ]:
def entrenar_pytorch(model, loader_train, loader_test, epochs=20, lr=0.1):
    model     = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    historia  = {'loss': [], 'acc': [], 'val_acc': []}

    for epoch in range(1, epochs+1):
        # ── Forward + Backward ──────────────────────────────────────────
        model.train()
        epoch_loss, correct, total = 0, 0, 0
        for Xb, yb in loader_train:
            Xb, yb = Xb.to(device), yb.to(device)

            optimizer.zero_grad()           # ① limpiar gradientes anteriores
            y_hat = model(Xb)               # ② forward
            loss  = criterion(y_hat, yb)    # ③ pérdida
            loss.backward()                 # ④ backward (autograd)
            optimizer.step()                # ⑤ actualizar pesos

            epoch_loss += loss.item() * len(yb)
            correct    += (y_hat.argmax(1) == yb).sum().item()
            total      += len(yb)

        # ── Evaluación ──────────────────────────────────────────────────
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for Xb, yb in loader_test:
                Xb, yb = Xb.to(device), yb.to(device)
                preds   = model(Xb).argmax(1)
                val_correct += (preds == yb).sum().item()
                val_total   += len(yb)

        loss_ep  = epoch_loss / total
        acc_ep   = correct / total
        val_acc  = val_correct / val_total
        historia['loss'].append(loss_ep)
        historia['acc'].append(acc_ep)
        historia['val_acc'].append(val_acc)
        print(f'Época {epoch:2d}  loss={loss_ep:.4f}  acc={acc_ep:.4f}  val_acc={val_acc:.4f}')

    return historia

In [ ]:
# ── Entrenar con CPU ──────────────────────────────────────────────────────
device = torch.device('cpu')
print(f'=== Entrenando en: {device} ===')
torch.manual_seed(42)
model_cpu = MLP_PyTorch([784, 256, 128, 10])
t0 = time.time()
hist_cpu = entrenar_pytorch(model_cpu, train_loader, test_loader)
t_cpu = time.time() - t0
print(f'Tiempo CPU: {t_cpu:.1f}s')

In [ ]:
# ── Entrenar con GPU (si está disponible) ─────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'=== Entrenando en: {device} ({torch.cuda.get_device_name(0)}) ===')
    torch.manual_seed(42)
    model_gpu = MLP_PyTorch([784, 256, 128, 10])
    t0 = time.time()
    hist_gpu = entrenar_pytorch(model_gpu, train_loader, test_loader)
    t_gpu = time.time() - t0
    print(f'Tiempo GPU: {t_gpu:.1f}s')
    print(f'\nSpeedup GPU vs CPU: {t_cpu/t_gpu:.1f}x')
else:
    print('GPU no disponible en este entorno.')
    hist_gpu = None

---
## Parte 3 — Comparación final

In [ ]:
# ── Gráfica comparativa ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, title in zip(axes, ['loss', 'val_acc'], ['Pérdida', 'Accuracy (Test)']):
    ax.plot(hist_np[metric],  'b-o', markersize=4, label='NumPy (CPU)')
    ax.plot(hist_cpu[metric], 'g-s', markersize=4, label='PyTorch (CPU)')
    if hist_gpu:
        ax.plot(hist_gpu[metric], 'r-^', markersize=4, label='PyTorch (GPU)')
    ax.set(title=title, xlabel='Época')
    ax.grid(alpha=.3); ax.legend()

plt.suptitle('Comparación: NumPy vs PyTorch CPU vs PyTorch GPU', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Tabla resumen ─────────────────────────────────────────────────────────
print('\n' + '='*55)
print(f'{"Método":<22} {"Acc. Test":>12} {"Tiempo (s)":>12}')
print('-'*55)
print(f'{"NumPy (CPU)":<22} {hist_np["val_acc"][-1]:>12.4f} {t_numpy:>12.1f}')
print(f'{"PyTorch (CPU)":<22} {hist_cpu["val_acc"][-1]:>12.4f} {t_cpu:>12.1f}')
if hist_gpu:
    print(f'{"PyTorch (GPU)":<22} {hist_gpu["val_acc"][-1]:>12.4f} {t_gpu:>12.1f}')
print('='*55)

---
## Visualización: errores del modelo
Ver qué dígitos confunde el modelo ayuda a entender sus limitaciones.

In [ ]:
# ── Ejemplos mal clasificados por el MLP NumPy ────────────────────────────
y_pred_np  = np.argmax(mlp_np.forward(X_test), axis=1)
errores    = np.where(y_pred_np != y_test)[0]

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for ax, idx in zip(axes.flat, errores[:16]):
    ax.imshow(X_test[idx].reshape(28, 28), cmap='gray')
    ax.set_title(f'Real:{y_test[idx]}  Pred:{y_pred_np[idx]}', fontsize=8)
    ax.axis('off')
plt.suptitle('Ejemplos mal clasificados (NumPy MLP)', fontsize=12)
plt.tight_layout()
plt.show()